# Project: Catan AI
***Dev log***
### Author: Alexander Makedonski

**Date:** 2026-03-13  
**Start time:** 16:00  
**End time:** 17:00  
**Session goal:** Setup project. Run a game  

Cloned catanatron (https://github.com/bcollazo/catanatron) as submodule. Added .gitignore.
Created venv: `py -m venv`. Then activate with: `.\venv\Scripts\Activate.ps1` if windows or `./venv/Scripts/activate` if linux. Installed developer dependencies: `pip install -e ./catanatron[web,gym,dev]`.
Ran a small simulation to check working dependencies.



**Date:** 2026-03-16  
**Start time:** 09:30  
**End time:** 12:00  
**Session goal:** Read documentation about catanatron. Find out a way to use for AI training  

The gym offers the possibility to generate data from games and use that as a point of study.
From what I gathered by skimming the entire documentation on https://docs.catanatron.com information from a game comes in 2 usable forms:  
1. One is the default observation of the board that is returned by `env.step()`/`env.reset()`. That observation is just a raw array of size `194 * N + 226` where `N` is the number of players on board  (source: https://catanatron.readthedocs.io/en/latest/catanatron.gym.envs.html#id2). 
2. The other one can be obtained with `catanatron.gym.board_tensor_features.create_board_tensor(game: Game, p0_color: Color, channels_first=False)` - A tensor of shape `(WIDTH=21, HEIGHT=11, CHANNELS=2*N+12)`, where `N` is the number of player in the game.

Thoughts: Skimmed over both variants to check what information is contained. Both seem appropriate, however variant 2:
- It contains a better representation of the board, but it alone is not enough to describe the current situation as it is missing information as IS_MOVING_ROBBER or IS_DISCARDING fields, that are contained in the first variant.
- It contains a lot of redundant information. A lot of the fields will stay 0 for the entire duration of the game. For example the tensor cell value that represents a road has N channels for every player, despite only one player ever being able to take control of it for the duration of the game.

The first arr will have to be used for sure (or some other methods) to get the needed information to feed the algorithm to play the game. Or maybe it can be used as enchancement for the second array, by adding more values to the tensor. But this might explode the space of possible inputs. Maybe can be used for building own observation features.

Notes: Seems like classic env is CatanatronEnv's default play is 1v1 with random player. (source: https://catanatron.readthedocs.io/en/latest/catanatron.gym.envs.html#module-catanatron.gym.envs.catanatron_env)

**Date:** 2026-03-25  
**Start time:** 15:00  
**End time:** 16:00  
**Session goal:** Understand the catanatron gym environment configuration settings. Try to setup DQN model to learn on 4 player map with randoms. Maybe graph the success rate.

Thoughts:
Ok, so one has to think how they will actually train the DQN. The DQN requires memory of previous (recent) experiences to be fed to the AI with a reward value. However it is hard to accertain what that reward value will be during the game and the moves.
1. Rewarding at the end of the game. I can map the not possible in the current state actions the DQN tries to make as bad, and some big negative value. That might not be a good idea, since that might discourage it from making moves that are possible in rare occasions. So it shouldn't be a big negative reward, but something small, like -0.001. But what about actions that it can make? One way is to map them to a reward value after the game is finished. If the DQN wins, it maps them to possitive values, negative otherwise - That doesn't sound good, because the agent doesn't know which action actually led it to winning the game so it might lead to something unstable.
2. Rewarding troughout the game. Give points for gaining resources, take points away for discarding, give points for building, getting longest road, largest army. Small rewards for small actions. And one big reward in the end for the win.

Actions:
Created some helper functions. More experiments required. 

**Date:** 2026-03-26  
**Start time:** 11:00  
**End time:** 14:30  
**Session goal:** Finished the setup of the DQN model.  

Thoughts:
Seems like yesterday there wasn't enough time to implement DQN, so I will try to do it now.
Actions:
Found a way to extract all the features of the game and make a named map for the observation. Create helpers with cache. Craeted reward function, memory for the DQN, the DQN itself and select_action function that masks unplayable actions. Created the q function for backprop. Needs testing and graphing to see how it works.


**Date:** 2026-04-18  
**Start time:** 14:30  
**End time:** 15:30  
**Session goal:** Run games with the DQN model. Excract some statistics for visualisation.  

Thoughts:
After making a  way to collect some information, I ran 2K games in about an hour. Which is really slow. Will have to find a way to upgrade the speed of the training. Also coded some way to extract info after the end of the game to see if the model is doing what it is supposed to do. Will have to update the reward function prbobably. Also maybe tweak the model and start smaller? Maybe even increase memory.
Actions:
Created some util functions to extract stats. Ran 2K games.

**Date:** 2026-04-19  
**Start time:** 14:30  
**End time:** 16:30  
**Session goal:** Optimize training. Try changing some settings.  

Thoughts:
Optimized dqn training. Now can do 1000 games in 30 minutes. Sadly training is not doing good, the model does not learn how to play the game, despite playing against random palyers. Changed weights of the mode to smaller model. Still not doing good. Started logging length of games, total_loss and reward for the game. Seems like dqn does fare well in this environment. Maybe because rewards are too sparse. Maybe I should try using the defalt reward function provided by the catanatron gym, since I am using my own now. Will probably try to define some unit tests to see if all the information saved for the model to train is acceptable.
Actions:
Optimized model training, by storing tensors directly in memory instead of transforming lists to tensors anytime they are batched for training. Also made tensors for actions masks for easier calculation later instead of an entire for cycle. Ran more trainings.

**Date:** 2026-04-21  
**Start time:** 14:00  
**End time:** 17:00  
**Session goal:** Change settings, try to achieve better training results.  

Thoughts:
Honestly, idk why DQN is not reaching at least somewhat good results. It appears to achieve randomness. Will probably increase memory size, play a bit with epsilon and other settings to see if anything changes. Maybe make the model more complex, since the game is really complex too.
Actions:
Changed the reward function. Increased model size. Played a bit with constants. Small fixes all around the different functions. Still no good results. Normalised observations by dividing by 50 at the start of each episode. Increased target network update frequency.

**Date:** 2026-04-21  
**Start time:** 17:00  
**End time:** 20:00  
**Session goal:** Implement Dueling DQN to improve learning stability.  

Thoughts:  
The standard DQN seems to be stuck at near-random performance. One possible reason is that during most steps the agent is making moves that have little direct impact on the outcome (e.g. ending turns, moving robber), so the advantage of one action over another is hard to distinguish. Dueling DQN separates the estimation of state value V(s) from per-action advantages A(s,a), which should help the model learn that some states are just inherently bad regardless of action taken. This decomposition may help with the sparse reward problem. Considering to try and setup the model against itself. Maybe the random oponents are too chaotic to reach some form of convergence.

Actions:  
Converted the DQN to a Dueling DQN. The model (Linear 512 -> ReLU -> Linear 512 -> ReLU -> Linear 256 -> ReLU) feeds into two separate heads: a value stream (256 -> 1) and an advantage stream (256 -> num_actions). The Q-values are combined as Q(s,a) = V(s) + (A(s,a) - mean_a(A(s,a))), which keeps the advantage mean-zero. Ran 10K episodes with memory=100k, epsilon decaying over 1M steps. Target network synced every 5 episodes. Stats logged to dqn_stats_dueling_model.json. Results still need more analysis however off the top observations the model does poorly against the random oponent.

**Date:** 2026-04-30 
**Start time:** 16:00  
**End time:** 20:00  
**Session goal:** Writing tests.  

Thoughts:
After looking at the agent performance I noticed that the loss never really reduces. Which signified to me that something is really off in the learning of the agent. I decided that unit tests are in order to check if the information fed to the agent is proper. After creating the test  I notcied that some of them weren't passing. Which is not great, will have to take a look another time.

Actions:
Wrote unit tests about: reward_function; valid_actions_to_mask; ReplayMemory; some DQN; Some optimize_model; reward_function for road points; create_game_stats; and some smoke test for training.

**Date:** 2026-05-06 
**Start time:** 12:00  
**End time:** 18:00  
**Session goal:** Bug fix - anything was using hardcoded player  

Thoughts:
Found a pretty disgusting bug today. It took me a lot of time to understand why the tests weren't passing. And also why they weren't passing by pure chance. The `reward_function` was hardcoding `"P0_"` as the player state key prefix everywhere - things like `game.state.player_state["P0_ACTUAL_VICTORY_POINTS"]`, `"P0_ROADS_AVAILABLE"`, etc. The actual correct way to get the key for a given color is via `player_key(game.state, color)`. So the agent was learning NOTHING, since it was 25% chance for the reward function to look at the agent.
Also cleaned up `reward_function` while I was in there - it had a `prints` debug parameter with a bunch of `if prints: print(...)` lines sprinkled in for road-building intermediate values. That was clearly leftover from debugging and had no business being in there anymore. Removed it all.
This means that the ENTIRE tiem spent teaching agents was useless. Will have to restart

Actions:  
Introduced the `player_key` function to the entire codebase. Refactored every function to use `player_key(game.state, agent_color)` instead of hardcoded `"P0_"` prefix. Updated some tests.

**Date:** 2026-05-07 
**Start time:** 12:00  
**End time:** 14:30  
**Session goal:** Try and train a DQN now  

Thoughts:
After fixing the bug, I tried training the DQN again. Added some other tests to make sure nothing else is broken. The dqn however did not do well, again. Will have to check if there are other bugs. It is not lowering its overall loss at all. Which to me again points at another bug in the training code.

Actions:  
Added some new tests. Trained another DQN.

**Date:** 2026-06-05  
**Start time:** 12:00  
**End time:** 15:00  
**Session goal:** Implement PPO + GAE  

Thoughts:  
Even after fixing the player_key bug, the DQN is still doing good as expected. The loss curve is doing good but the winrate is still 50% in 1v1 scenario. It is just bouncing around with no clear downward trend. I suspect the core issue is credit assignment - Catan games last hundreds of turns and the only meaningful reward signal comes at the very end or at building cities/settlements which happens rarely. DQN replays individual transitions, so the connection between an early good move and the eventual win is almost impossible to learn. Also, most of the moves produce no reward at all, so the agent probably tries to predict almost 0 on everything, since it does nothing. The Q-value updates also tend to be noisy with off-policy methods as read on the internet.

PPO seems like a better fit. It is an on-policy actor-critic method that processes complete episode rollouts, so the full sequence of decisions that led to an outcome is available during each update. GAE (Generalized Advantage Estimation) lets me interpolate between high-variance Monte Carlo returns and high-bias estimates by tuning the lambda parameter, which should give more meaningful training signal across the long time horizons of a Catan game. Worth trying.

Actions:  
Implemented `PPOActor` and `PPOCritic` networks. Implemented `compute_gae` for advantage estimation. Implemented `ppo_update` with the clipped surrogate objective, critic MSE loss, and entropy bonus. Wired everything up in `ppo_training.py`.

**Date:** 2026-06-06  
**Start time:** 13:00  
**End time:** 16:00  
**Session goal:** Test and run PPO training  

Thoughts:  
Ran the PPO for the first time. Results are not good. The actor loss oscillates without any meaningful reduction and the win rate is still 50%, so it's still just random play. Hard to tell if this is a bug or just the algorithm needing more time, but after 20K episodes I thought it would be enough and start converging at least a little. Tried a couple of different parameter combinations - different learning rates, entropy coefficients - nothing made a substantial difference. Could be the same fundamental problem as the DQN: the reward signal is too sparse and too delayed for the policy to figure out what matters. Or there is still something wrong in the code that I am not seeing yet.

Actions:  
Ran PPO training for several thousand episodes. Experimented with learning rate, rollout buffer size, and entropy coefficient. Logged actor loss, critic loss, and game stats. No improvement observed.

**Date:** 2026-06-13  
**Start time:** 15:00  
**End time:** 17:00  
**Session goal:** Build a dashboard to better visualise training statistics  

Thoughts:  
Decided to build a Streamlit dashboard to visualise the logged training data. I hope that seeing the curves plotted might reveal something that is not obvious from the numbers. Spent the session building it and exploring the existing training data. Did not notice anything particularly revealing. The curves are flat (where as they should be rising) and there are no obvious anomalies that point to a specific bug.

Actions:  
Built a Streamlit dashboard in `src/dashboard.py` to visualise training stat JSON files. Added charts anything meaningfull. Loaded existing DQN and PPO stat files and explored the data.

**Date:** 2026-06-14  
**Start time:** 11:00  
**End time:** 18:00  
**Session goal:** Tweak reward function and hyperparameters. Run more training.  

Thoughts:  
Spent the day trying to brute-force the problem by changing anything that wouldn't outright ruin the algorithms. Updated the reward function with additional intermediate signals - building roads, settlements, and cities - to try to reduce reward sparsity. Tried different learning rates, batch sizes, discount factors (`GAMMA`), and network sizes for both DQN and PPO. Ran a bunch of training runs in parallel for both models. None of it worked. The stats look the same as before: reducing loss for the DQN from 1.8 to 0.2 then flat, flat loss for the PPO, win rate around what a random agent would produce - 50%. It is becoming difficult to tell whether the problem is the algorithm, the reward, the architecture, or something more fundamental. Feeling demoralized. Also a bit insane for even trying.

Actions:  
Modified reward function with additional intermediate shaping signals. Adjusted training hyperparameters across multiple runs. Trained additional DQN and PPO instances. Logged and compared results.

**Date:** 2026-06-20  
**Start time:** 12:00  
**End time:** 18:00  
**Session goal:** Seek external input. Try suggested approaches.  

Thoughts:  
At this point I have been stuck for weeks with both models showing no sign of improvement. Decided to contact two TA's, one of the deep machine learning course and one from the tensorflow course (which is actually pytorch, but that is fine). The first TA did not respond, but a second one got back to me. Their suggestions were to try a significantly higher learning rate and to switch to the default reward function provided by the catanatron gym (which gives +1 on win and -1 on loss) instead of the hand-engineered one. Since that might give wrong signals to the training algorithm and teach i tto maximaze not winning but pleasing the reward. Both are reasonable ideas - a higher learning rate might help escape flat regions in the gradient descent. Applied both. Neither made any difference. Loss is still flat, win rates unchanged. Started looking at other potential explanations: self-play of the algorithms, simplifying the action space. Nothing obviously promising.

Actions:  
Contacted course teaching assistants. Applied suggested higher learning rate and default win/loss reward function for both DQN and PPO. Ran training runs with updated settings. Logged and compared results against previous runs. No improvement found.

**Date:** 2026-06-21  
**Start time:** 10:00  
**End time:** 17:00  
**Session goal:** Try to diagnose why models are not learning.

Thoughts:  
At this point I started seriously considering that the game might just be too complex for reinforcement learning with the current setup. The observation space is huge, the action space is large. High bias feels like a good explanation - maybe the models simply do not have enough capacity to learn a good policy for Catan. Increased the network size significantly and ran training again. Still nothing.

Then I noticed something odd in the training statistics: the average game length was steadily decreasing as training progressed. When I thought about it - the only way games get shorter is if someone is winning faster. The random players are not getting better, so it has to be the agent. But the logged win rate was stuck at around 50%, which is what random play looks like in a 2-player game. That made no sense. So i decided to check the log function `create_game_stats` and I noticed it was calling `player_key(game.state, game.state.colors[0])`. It was always reading the stats for whichever player happened to be at index 0 in the color list, which is one of the random opponents, not the BLUE agent (which is our agent. Always marked with blue). Every win rate, VP count, road count logged over weeks of training belonged to a random player. The models were winning all along and the stats were lying the entire time. xDDDD

Deleted all the corrupted stat files. Retrained both models from scratch with the correct stats tracking. DQN with the engineered reward function reached **85% win rate** against 1 random player. PPO with the default catanatron win/lose reward reached **76% win rate**. Both models are actually working. I feel like I dropped a rock from my shoulders.

Actions:  
Investigated game-length trend in training stats. Identified the bug in `create_game_stats` always reading `colors[0]` instead of `Color.BLUE`. Fixed the function to use the correct player key. Added unit tests covering this exact invariant (enemy state changes must not appear in main-player stats, and vice versa). Deleted all previously logged stats. Retrained DQN and PPO. Confirmed working models with correct win rates.

**Date:** 2026-06-22  
**Start time:** 18:00  
**End time:** 21:00  
**Session goal:** Create reinforce learning algorithm.

Thoughts:  
Try and add reinforce learning algorithm to see if it is going to work. Simple is best after all. Also try to train smaller implementations of the networks, to maybe reduce overfitting? After training a smaller DQN I noticed that it is winning mor games generally than the bigger model after more training. Will have to check if more training makes the bigger model better at the game too. The difference between 5K and 15K episodes is about 10% winrate. Which is a lot, but still.

Actions:  
Created reinforce learning algorithm. Trained some more models.

**Date:** 2026-06-23  
**Start time:** 18:00  
**End time:** 22:00  
**Session goal:** Create self-playing algorithm. Run the best models through it.

Thoughts:  
I wanna create an algorithm where the agents play themselves. That would allow for more learning and exploitation of the environment. Trained some reinforcement learning. It didn't work well. 25% winrate against random which is weird. Will not investigate further, since the time is limited. Might be because of some bug in the training. Training on the DQN's to selfplay however turned out really good. Reached 90% winrate with one of them (against weighted random player) after training, which is really good. Trained a D3QN (Dueling Double Deep Q-Network) which is doing really good also.

Actions:  
Trained reinforce algorithm. Trained some self playing DQNs. Trained a D3QN. Create `ModelPlayer` which wraps a model as a catanatron player to be inserted into the game environment. That way we can insert model to play against itself and get better.